# Bronze Layer: Product Info (Auto Loader)

**Pattern**: Auto Loader (Streaming Ingestion)  
**Trigger**: AvailableNow (Incremental Batch)

In [0]:
from pyspark.sql.functions import current_timestamp

source_path = "/Volumes/workspace/bronze/raw_sources/source_crm/"
target_table = "workspace.bronze.crm_prd_info_cdc"
checkpoint_path = "/Volumes/workspace/bronze/raw_sources/checkpoints/crm_prd_info"
schema_path = "/Volumes/workspace/bronze/raw_sources/checkpoints/crm_prd_info_schema"

query = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true") # Fixed: removed cloudFiles prefix
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("pathGlobFilter", "prd_info.csv")
    .load(source_path)
    .withColumn("ingestion_timestamp", current_timestamp())
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()